In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import nltk
from collections import Counter

nltk.download('punkt')

# =========================
# 1. LOAD DATA
# =========================

dataset = load_dataset("squad")
data = dataset["train"]

pairs = [(item["question"], item["answers"]["text"][0]) for item in data]
pairs = pairs[:5000]

print("Total pairs:", len(pairs))

# =========================
# 2. TOKENIZER
# =========================

def tokenize(text):
    return nltk.word_tokenize(text.lower())

# =========================
# 3. VOCAB
# =========================

counter = Counter()
for q, a in pairs:
    counter.update(tokenize(q))
    counter.update(tokenize(a))

PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"

vocab_size = 15000
most_common = counter.most_common(vocab_size - 4)

idx2word = [PAD, UNK, SOS, EOS] + [w for w, _ in most_common]
word2idx = {w: i for i, w in enumerate(idx2word)}

print("Vocab:", len(word2idx))

# =========================
# 4. ENCODE
# =========================

MAX_LEN = 30

def encode(text):
    tokens = [SOS] + tokenize(text) + [EOS]
    ids = [word2idx.get(t, word2idx[UNK]) for t in tokens]

    if len(ids) < MAX_LEN:
        ids += [word2idx[PAD]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

# =========================
# 5. DATASET
# =========================

class QADataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, a = self.pairs[idx]
        return torch.tensor(encode(q)), torch.tensor(encode(a))

dataset = QADataset(pairs)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# =========================
# Positional Encoding
# =========================

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=100):
        super().__init__()

        encoding = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, embed_size, 2) * (-torch.log(torch.tensor(10000.0)) / embed_size)
        )

        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term)

        self.encoding = encoding.unsqueeze(0)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1), :]

# =========================
# Multi Head Attention + Mask
# =========================

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, heads=4):
        super().__init__()

        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)

        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, query, key, value):
        N, query_len, _ = query.shape
        _, key_len, _ = key.shape

        V = self.values(value)
        K = self.keys(key)
        Q = self.queries(query)
    
        V = V.view(N, key_len, self.heads, self.head_dim)
        K = K.view(N, key_len, self.heads, self.head_dim)
        Q = Q.view(N, query_len, self.heads, self.head_dim)
    
        energy = torch.einsum("nqhd,nkhd->nhqk", Q, K)
    
        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=3)
    
        out = torch.einsum("nhql,nlhd->nqhd", attention, V)
    
        out = out.reshape(N, query_len, self.embed_size)
    
        return self.fc_out(out)

# =========================
# Encoder
# =========================

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos = PositionalEncoding(embed_size)
        self.attn = MultiHeadAttention(embed_size)
        self.norm = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos(x)

        attn = self.attn(x, x, x)
        x = self.norm(x + attn)

        return x

# =========================
# Decoder
# =========================

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos = PositionalEncoding(embed_size)

        self.self_attn = MultiHeadAttention(embed_size)
        self.cross_attn = MultiHeadAttention(embed_size)
        
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.fc = nn.Linear(embed_size, vocab_size)

        self.ff = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size)
        )
        
        self.norm3 = nn.LayerNorm(embed_size)

    def forward(self, x, enc_out):
        x = self.embedding(x)
        x = self.pos(x)

        # Self Attention
        attn1 = self.self_attn(x, x, x)
        x = self.norm1(x + attn1)
        
        # Cross Attention
        attn2 = self.cross_attn(x, enc_out, enc_out)
        x = self.norm2(x + attn2)
        
        # Feed Forward
        ff = self.ff(x)
        x = self.norm3(x + ff)

        return self.fc(x)

# =========================
# Seq2Seq
# =========================

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_size):
        super().__init__()

        self.encoder = Encoder(vocab_size, embed_size)
        self.decoder = Decoder(vocab_size, embed_size)

    def forward(self, src, trg):
        enc_out = self.encoder(src)
        out = self.decoder(trg, enc_out)
        return out

model = Seq2Seq(len(word2idx), 64)

# =========================
# TRAINING
# =========================

criterion = nn.CrossEntropyLoss(ignore_index=word2idx[PAD])
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 20

for epoch in range(epochs):
    total_loss = 0

    for questions, answers in loader:
        optimizer.zero_grad()

        outputs = model(questions, answers[:, :-1])
        target = answers[:, 1:]

        outputs = outputs.reshape(-1, outputs.shape[-1])
        target = target.reshape(-1)

        loss = criterion(outputs, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")

# =========================
# PREDICT
# =========================

def predict_sentence(text, max_len=12, beam_width=3):
    model.eval()

    src = torch.tensor([encode(text)])

    sequences = [([word2idx[SOS]], 0.0)]

    with torch.no_grad():
        for _ in range(max_len):
            all_candidates = []

            for seq, score in sequences:
                trg = torch.tensor([seq + [word2idx[PAD]]*(MAX_LEN-len(seq))])

                output = model(src, trg)
                logits = output[0, len(seq)-1]

                probs = torch.log_softmax(logits, dim=0)

                topk = torch.topk(probs, beam_width)

                for i in range(beam_width):
                    word_idx = topk.indices[i].item()

                    
                    if word_idx in [word2idx[PAD], word2idx[SOS]]:
                        continue

                    new_seq = seq + [word_idx]
                    new_score = score + topk.values[i].item()

                    all_candidates.append((new_seq, new_score))

            sequences = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:beam_width]

    best_seq = sequences[0][0]

    result = []
    for idx in best_seq:
        word = idx2word[idx]

        if word in [SOS, PAD]:
            continue
        if word == EOS:
            break

        result.append(word)

    return " ".join(result) if result else "..."

# =========================
# TEST
# =========================

print("\nExample Predictions:\n")
print(predict_sentence("Who is he?"))
print(predict_sentence("What is Python?"))
print(predict_sentence("Where is New York?"))
print(predict_sentence("When did the war start?"))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Total pairs: 5000
Vocab: 8198
Epoch 1, Loss: 1070.61
Epoch 2, Loss: 895.37
Epoch 3, Loss: 818.24
Epoch 4, Loss: 756.18
Epoch 5, Loss: 699.10
Epoch 6, Loss: 643.83
Epoch 7, Loss: 587.39
Epoch 8, Loss: 532.27
Epoch 9, Loss: 478.70
Epoch 10, Loss: 425.55
Epoch 11, Loss: 374.78
Epoch 12, Loss: 325.16
Epoch 13, Loss: 278.33
Epoch 14, Loss: 234.71
Epoch 15, Loss: 193.57
Epoch 16, Loss: 157.65
Epoch 17, Loss: 124.69
Epoch 18, Loss: 95.44
Epoch 19, Loss: 72.13
Epoch 20, Loss: 52.28

Example Predictions:

metres
bored
...
...
